# Analyze Drudge hyperlinks

In [138]:
import re
import sys
from pathlib import Path
from urllib.parse import urlparse

In [139]:
import storysniffer

In [140]:
import numpy as np
import pandas as pd
import altair as alt

In [141]:
this_dir = Path("__file__").parent.absolute()
sys.path.append(this_dir.parent)
sys.path.append(str(this_dir.parent / "newshomepages"))

In [142]:
import altair_theme

In [143]:
alt.themes.register('palewire', altair_theme.theme)
alt.themes.enable('palewire')

ThemeRegistry.enable('palewire')

In [144]:
extracts_dir = this_dir.parent / "extracts" / "csv"

In [145]:
analysis_dir = this_dir.parent / "_analysis"

Read in the sample data

In [146]:
df = pd.read_csv(
    extracts_dir / "drudge-hyperlinks-sample.csv",
    usecols=[
        'handle',
        'file_name',
        'date',
        'text',
        'url',
    ],
    dtype=str,
    parse_dates=["date"]
)

In [147]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62805 entries, 0 to 62804
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   handle     62805 non-null  object        
 1   file_name  62805 non-null  object        
 2   date       62805 non-null  datetime64[ns]
 3   text       62535 non-null  object        
 4   url        62804 non-null  object        
dtypes: datetime64[ns](1), object(4)
memory usage: 2.4+ MB


In [148]:
df.sort_values("date").tail()

,handle,file_name,date,text,url
62636,DRUDGE,drudge-2022-08-31T06:59:54.062655-04:00.hyperl...,2022-08-31,PEOPLE,http://www.people.com
62637,DRUDGE,drudge-2022-08-31T06:59:54.062655-04:00.hyperl...,2022-08-31,POLITICO,http://www.politico.com/
62638,DRUDGE,drudge-2022-08-31T06:59:54.062655-04:00.hyperl...,2022-08-31,RADAR,https://radaronline.com/
62640,DRUDGE,drudge-2022-08-31T06:59:54.062655-04:00.hyperl...,2022-08-31,REASON,https://reason.org/
62804,DRUDGE,drudge-2022-08-31T06:59:54.062655-04:00.hyperl...,2022-08-31,PRIVACY POLICY,/privacy/


Guess links with `storysniffer`

In [149]:
links_df = df.groupby(["text", "url"]).size().rename("n").reset_index()

In [150]:
sniffer = storysniffer.StorySniffer()

In [151]:
links_df.describe()

,n
count,7094.000000
mean,8.815055
std,39.944921
min,1.000000
25%,1.000000
50%,2.000000
75%,3.000000
max,258.000000


In [152]:
links_df['is_story'] = links_df.apply(lambda x: sniffer.guess(x['url'], text=x['text']), axis=1)

In [153]:
links_df.is_story.value_counts()

True     6835
False     259
Name: is_story, dtype: int64

In [154]:
links_df.is_story.value_counts(normalize=True)

True     0.96349
False    0.03651
Name: is_story, dtype: float64

In [155]:
links_df.to_csv("./drudge-hyperlinks-storysniffer-guesses.csv", index=False)

Make some manual fixes

In [156]:
blacklist = [
    "/privacy/",
]

In [157]:
links_df.loc[
    links_df.url.isin(blacklist),
    'is_story'
] = False

In [158]:
substack_pattern = ".(substack|theankler|commonsense).(com|news)/p/"

In [159]:
links_df.loc[
    links_df.url.str.contains(substack_pattern),
    'is_story'
] = True

/tmp/ipykernel_1839511/673093257.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  links_df.url.str.contains(substack_pattern),


In [160]:
time_pattern = "^https://time.com/\d{5,}/*"

In [161]:
links_df.loc[
    links_df.url.str.contains(time_pattern),
    'is_story'
] = True

In [162]:
studyfinds_pattern = "^https://www.studyfinds.org/*.{5,}"

In [163]:
links_df.loc[
    links_df.url.str.contains(studyfinds_pattern),
    'is_story'
] = True

In [164]:
bbc_pattern = "^https://www.bbc.com/news/*.{5,}"

In [165]:
links_df.loc[
    links_df.url.str.contains(bbc_pattern),
    'is_story'
] = True

In [166]:
jpost_pattern = "^https://www.jpost.com/breaking-news/*.{5,}"

In [167]:
links_df.loc[
    links_df.url.str.contains(jpost_pattern),
    'is_story'
] = True

In [168]:
braint_pattern = "^https://www.braintomorrow.com/*.{5,}"

In [169]:
links_df.loc[
    links_df.url.str.contains(braint_pattern),
    'is_story'
] = True

Knock out anything that appears most of the time

In [170]:
n = len(df.file_name.unique())

In [171]:
links_df.loc[
    links_df.n >= n * .5,
    'is_story',
] = False

In [172]:
links_df.to_csv("./drudge-hyperlinks-storysniffer-tweaks.csv", index=False)

In [173]:
links_df[
    (links_df.n < 40) &
    (links_df.is_story == False)
].sort_values("url").head(5)

,text,url,n,is_story
3451,LIVE: TEMP MAP...,http://hp2.wright-weather.com/icons/us_heat.gif,14,False
3446,LIVE HEAT MAP...,http://hp2.wright-weather.com/icons/us_heat.gif,8,False
3450,LIVE: TEMP MAP,http://hp2.wright-weather.com/icons/us_heat.gif,2,False
4494,PARTICIPANTS...,https://bilderbergmeetings.org/press/press-rel...,1,False
3622,MAP...,https://earthquake.usgs.gov/earthquakes/eventp...,5,False


Tally domains

In [174]:
links_df['domain'] = links_df.url.apply(lambda x: urlparse(x).netloc)

In [175]:
links_df[links_df.is_story].domain.value_counts().reset_index().head(10)

,index,domain
0,www.msn.com,806
1,apnews.com,507
2,www.wsj.com,372
3,news.yahoo.com,322
4,dnyuz.com,310
5,www.dailymail.co.uk,297
6,www.cnbc.com,216
7,www.the-sun.com,211
8,www.cnn.com,208
9,nypost.com,207


In [193]:
story_df = links_df[links_df.is_story].copy()

In [194]:
def is_trump(row):
    token_list = [t.lower() for t in row['text'].split()]
    if 'trump' in token_list:
        return True
    elif 'donald' in token_list:
        return True
    elif 'don' in token_list:
        return True
    else:
        if 'trump' in row['url']:
            return True
    return False

In [195]:
story_df['is_trump'] = story_df.apply(is_trump, axis=1)

In [196]:
story_df.is_trump.value_counts()

False    6206
True      648
Name: is_trump, dtype: int64

In [197]:
trump_df = story_df[story_df.is_trump].copy()

In [198]:
date_df = df[['url', 'date']].sort_values(["url", "date"], ascending=[True, True]).drop_duplicates("url", keep="first")

In [199]:
trump_df = trump_df.merge(date_df, on="url").sort_values("date", ascending=False)

In [201]:
trump_df.head()

,text,url,n,is_story,domain,is_trump,date
174,Demands to Be Reinstated as President...,https://www.mediaite.com/trump/trump-re-ups-hi...,1,True,www.mediaite.com,True,2022-08-31
34,'Those Records Do Not Belong to Him'...,https://dnyuz.com/2022/08/30/justice-departmen...,1,True,dnyuz.com,True,2022-08-31
531,TRUMP CAUGHT HOARDING NATION'S SECRETS,https://apnews.com/article/mar-a-lago-governme...,1,True,apnews.com,True,2022-08-31
182,Documents at Mar-a-Lago Were Moved and Hidden ...,https://news.yahoo.com/u-justice-dept-responds...,1,True,news.yahoo.com,True,2022-08-31
508,Shares barrage of QAnon content...,https://www.nbcnews.com/politics/donald-trump/...,2,True,www.nbcnews.com,True,2022-08-30


In [207]:
list(trump_df.text.unique())[:30]

['Demands to Be Reinstated as President...',
 "'Those Records Do Not Belong to Him'...",
 "TRUMP CAUGHT HOARDING NATION'S SECRETS",
 'Documents at Mar-a-Lago Were Moved and Hidden as Feds Sought Them...',
 'Shares barrage of QAnon content...',
 'DEMENTING DON DEMANDS NEW ELECTION?',
 'Legal team advances broad view of presidential powers at Mar-a-Lago...',
 'Legal team advances broad view of presidential powers in Mar-a-Lago probe...',
 'The Don  boasted  he had intelligence about Macron sex life...',
 "'TRUTH' barred from GOOGLE store over moderation concerns...",
 'Republicans forced into strained defense...',
 'MORE WARNINGS OF UNREST',
 'MUST BE HELD IMMEDIATELY!',
 'Trump boasted  he had intelligence about Macron sex life...',
 'Secret Service agent for Trump retires after Jan. 6 scrutiny...',
 "What's Behind War With Intel Community?",
 'INDICTMENT WATCH INTENSIFIES',
 'FBI completes review of docs...',
 'Midterms Turn Toxic...',
 'Legal Team Scrambles to Find Argument..',
 'Ben 